Ссылка на урок: [Спринт 2/5: 2 RAG: от сырых данных до точных ответов → Тема 3/4: Архитектура RAG → Урок 6/6: Сборка RAG-пайплайна с LangChain]( https://practicum.yandex.ru/learn/ai-engineer/courses/99ad0dee-d9e6-4da1-b921-64e7e9653a4e/sprints/1104074/topics/b36d9f5f-3f26-4fbb-bd65-71d33c4523ef/lessons/827f1ac2-fe38-474d-843b-7f4ea53670d3/#8a453b5b-ef1e-402b-93fa-8951c603da30 )

In [1]:
from collections.abc import Iterator

from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document
from pypdf import PdfReader


class PypdfPageLoader(BaseLoader):
    """Читает PDF постранично и возвращает объекты Document."""

    def __init__(self, file_path: str):
        self.file_path = file_path

    def lazy_load(self) -> Iterator[Document]:
        """Читает страницы лениво, сохраняя источник и номер страницы."""
        reader = PdfReader(self.file_path)
        file_metadata = {
            key.removeprefix("/").lower(): value
            for key, value in (reader.metadata or {}).items()
        }

        for page_number, page in enumerate(reader.pages):
            yield Document(
                page_content=page.extract_text() or "",
                metadata={
                    **file_metadata,
                    "source": self.file_path,
                    "page": page_number,
                    "total_pages": len(reader.pages),
                },
            )

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Пропуск 1: загрузите velta_guide.pdf — получите список Document
loader = <...>
docs = <...>
print("Страниц:", len(docs))
print("Метаданные:", docs[0].metadata["source"], "| стр.", docs[0].metadata["page"])

# Пропуск 2: нарежьте документы на чанки — размер 800, перекрытие 120
splitter = <...>
chunks = <...>
print("Чанков:", len(chunks))

# Эмбеддер — multilingual-e5-base.
# Сама обёртка не добавит префиксы e5 — пробрасываем их явно
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True, "prompt": "passage: "},
    query_encode_kwargs={"normalize_embeddings": True, "prompt": "query: "},
)

# Пропуск 3: соберите базу из чанков; метрика — косинус
vectorstore = <...>
# У обёртки нет публичного счётчика записей — заглядываем во внутреннюю коллекцию Chroma
print("Записей в базе:", vectorstore._collection.count())

# Проверка поиска знакомым запросом: метод вернёт пары (чанк, расстояние) —
# чем меньше dist, тем ближе
for doc, dist in vectorstore.similarity_search_with_score(
        "что означает код ошибки E-52", k=3):
    print(f"dist={dist:.3f} | стр. {doc.metadata['page']} | "
          f"{doc.page_content[:60]!r}")